# 客户阶段整理：地区规则、订单关联与数量核对

本次独立复现客户阶段：原始客户记录 → 工作副本 → 订单关联 → 州城市分组 → 控制总额。

先保存第十份 Notebook，再从新内核逐格运行。代码的注释会标明每张表的粒度，以及哪些字段用于连接、哪些用于分组。

客户表一行一个 customer_id；客户标识观察表一行一个 customer_unique_id；地区订单表一行一个州+城市组合。

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'
src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from customers_cleaning import (
    LOCATION_FLAG_COLS, STATE_CODES, clean_customers,
    check_customer_locations, customer_field_changes, join_orders_customers,
    summarize_city_orders, summarize_customer_identity,
    reconcile_customer_totals, summarize_customer_metrics
)

def read_raw(filename):
    # 邮编和 ID 保留字符串；01151 的前导零不能丢。
    # 原始客户与订单表保留，后面由函数返回新表。
    return pd.read_csv(
        data_dir / filename, dtype=str, encoding='utf-8-sig',
        keep_default_na=False, na_values=['']
    )

raw_customers = read_raw('olist_customers_dataset.csv')
raw_orders = read_raw('olist_orders_dataset.csv')
print('原始客户记录数：', len(raw_customers))
print('原始订单数：', len(raw_orders))

原始客户记录数： 99441
原始订单数： 99441


## 1. 保留客户记录，规范地区并标记问题

clean_customers 检查 customer_id 主键和 customer_unique_id 非空，返回完整工作副本。customer_unique_id 可以重复，不按它删除记录。

城市、州、邮编只做首尾空白和大小写规范，邮编不自动补零。州代码会同时检查格式和 IBGE 参考成员关系；参考来自 [IBGE 官方接口](https://servicodados.ibge.gov.br/api/v1/localidades/estados?orderBy=nome)，本地快照让脚本无需联网运行。

In [2]:
# 输出仍是一行一个 customer_id 的客户记录，不是去重后的客户人数表。
customers_clean = clean_customers(raw_customers)

display(check_customer_locations(raw_customers, customers_clean))
display(customers_clean[LOCATION_FLAG_COLS + [
    'flag_customer_location_needs_review'
]].sum())

print('客户记录数：', len(customers_clean))
print('不同跨订单客户标识数：', customers_clean['customer_unique_id'].nunique())
print('参考州代码数量：', len(STATE_CODES))
print('实际发生变化的字段条数：',
      len(customer_field_changes(raw_customers, customers_clean)))

# 查看保留前导零的邮编样本。这里没有转为数字。
customers_clean.loc[
    customers_clean['customer_zip_code_prefix'].str.startswith('0'),
    ['customer_zip_code_prefix', 'customer_city', 'customer_state']
].head(3)

,原始缺失数,非空但只有空白字符数,规范后变化记录数
customer_zip_code_prefix,0,0,0
customer_city,0,0,0
customer_state,0,0,0


flag_zip_format_invalid                0
flag_state_format_invalid              0
flag_state_unknown                     0
flag_city_missing                      0
flag_customer_location_needs_review    0
dtype: int64

客户记录数： 99441
不同跨订单客户标识数： 96096
参考州代码数量： 27
实际发生变化的字段条数： 0


,customer_zip_code_prefix,customer_city,customer_state
1,09790,sao bernardo do campo,SP
2,01151,sao paulo,SP
3,08775,mogi das cruzes,SP


## 2. 按客户记录 ID 关联订单

join_orders_customers 的连接键是 customer_id。先保证客户主键唯一、订单键非空且有匹配，再左连接。

当前数据中订单侧 customer_id 也唯一，所以延续课堂的一对一验证。关联结果一行一个订单；同一 customer_unique_id 在不同订单出现是允许的。

In [3]:
# 用工作表关联，让规范后的地区字段进入订单；两张原表不被覆盖。
orders_with_customers = join_orders_customers(raw_orders, customers_clean)

print('关联后订单行数：', len(orders_with_customers))
display(orders_with_customers['customers_match'].value_counts())

orders_with_customers[
    ['order_id', 'order_status', 'customer_id', 'customer_unique_id',
     'customer_zip_code_prefix', 'customer_city', 'customer_state']
].head()

关联后订单行数： 99441


customers_match
both          99441
left_only         0
right_only        0
Name: count, dtype: int64

,order_id,order_status,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,09195,santo andre,SP


## 3. 在哪里按“州+城市”聚合

summarize_city_orders 中的关键代码是：

```python
# eligible_orders 已经是一行一个订单。
# 两个分组键避免把不同州的同名城市合并。
city_orders = eligible_orders.groupby(
    ["customer_state", "customer_city"], as_index=False
).agg(
    order_count=("order_id", "size")  # 统计该组的订单行，得到该州城市的订单数
)
```

只要州和城市可用，就能参与这项统计；邮编格式问题不会自动让整笔订单退出州城市统计。地区不可用的订单仍留在完整关联表里，并单独报告排除数。

In [4]:
# 新表一行一个州+城市组合，行数减少是分组粒度变化。
city_orders = summarize_city_orders(orders_with_customers)

print('地区统计行数：', len(city_orders))
print('地区表订单计数合计：', city_orders['order_count'].sum())
print('不满足州城市分组条件的订单数：',
      (~orders_with_customers['can_group_by_city_state']).sum())

# 当前是全部历史订单的计数，尚未筛选正式购买状态或分析区间。
city_orders.head()

地区统计行数： 4310
地区表订单计数合计： 99441
不满足州城市分组条件的订单数： 0


,customer_state,customer_city,order_count
0,SP,sao paulo,15540
1,RJ,rio de janeiro,6882
2,MG,belo horizonte,2773
3,DF,brasilia,2131
4,PR,curitiba,1521


In [5]:
# 对照源订单数量及本批订单实际匹配的客户标识。
# 最后一行比较可分组订单数与地区表 order_count 合计。
customer_controls = reconcile_customer_totals(
    raw_orders, customers_clean, orders_with_customers, city_orders
)
customer_controls

,来源或预期,处理后,差额
订单行数,99441,99441,0
订单ID不同值数,99441,99441,0
匹配客户订单数,99441,99441,0
匹配的客户标识数,96096,96096,0
可按州城市分组的订单数,99441,99441,0


## 4. 同一客户标识有不同地区时，保留各条记录

customer_identity_summary 只观察每个 customer_unique_id 的记录数和地区字段差异，不用它替换按 customer_id 关联的客户表。

这些字段差异不能直接确定搬家等业务原因。将某一条地址覆盖到该客户的所有历史订单，会丢失订单各自对应的位置。

In [6]:
# 一行一个跨订单客户标识，仅用于观察；不生成“已复购”的标签。
customer_identity_summary = summarize_customer_identity(customers_clean)

display(pd.Series({
    '客户记录数': len(customers_clean),
    '跨订单客户标识数': len(customer_identity_summary),
    '对应多条客户记录的标识数': (
        customer_identity_summary['customer_record_count'].gt(1).sum()
    ),
    '对应多个州城市组合的标识数': (
        customer_identity_summary['city_state_count'].gt(1).sum()
    ),
    '对应多个州标签的标识数': (
        customer_identity_summary['state_count'].gt(1).sum()
    )
}))

customer_identity_summary.loc[
    customer_identity_summary['state_count'].gt(1)
].head()

客户记录数            99441
跨订单客户标识数         96096
对应多条客户记录的标识数      2997
对应多个州城市组合的标识数      122
对应多个州标签的标识数         39
dtype: int64

,customer_unique_id,customer_record_count,postcode_count,state_count,city_state_count
11934,1f90117a847636892e3c5bf569f2ac68,2,2,2,2
13620,2410195f6521688005612363835a2671,2,2,2,2
16656,2c45ab66a3dae52960147e76a35740ff,2,2,2,2
16707,2c6a91479a7dc00d8c9d650d8dee88ca,2,2,2,2
24297,408aee96c75632a92e5079eee61da399,2,2,2,2


## 5. 为什么格式通过，还要检查参考取值

下面只创建一条演示副本，将州设为 AA。它是两个大写字母，但不在已核对的州代码里。

演示不会改变原表或前面的工作表。观察三个字段：格式标记、未知代码标记、能否按州城市分组。

In [7]:
# 仅用于理解规则的临时副本，不写入正式结果。
demo_customers = raw_customers.iloc[[0]].copy()
demo_customers['customer_state'] = 'AA'
demo_clean = clean_customers(demo_customers)

demo_clean[
    ['customer_state', 'flag_state_format_invalid',
     'flag_state_unknown', 'can_group_by_city_state']
]

,customer_state,flag_state_format_invalid,flag_state_unknown,can_group_by_city_state
0,AA,False,True,False


In [8]:
# 汇总当前真实数据的基线，后面的演示副本没有参与计算。
metrics = summarize_customer_metrics(
    raw_customers, customers_clean, orders_with_customers,
    city_orders, customer_identity_summary
)

pd.Series({
    '客户记录数': metrics['customer_rows'],
    '客户标识数': metrics['customer_unique_id_unique'],
    '关联后订单数': metrics['all_order_rows'],
    '地区待核查记录数': metrics['location_review_rows'],
    '州城市组合数': metrics['city_state_order_groups'],
    '地区订单计数合计': metrics['orders_in_city_state_groups'],
    '跨州同名城市名称数': metrics['city_names_across_multiple_states']
})

客户记录数        99441
客户标识数        96096
关联后订单数       99441
地区待核查记录数         0
州城市组合数        4310
地区订单计数合计     99441
跨州同名城市名称数      163
dtype: int64

## 独立导出与下一步

本 Notebook 在内存中复现。从项目目录运行 python .\src\06_clean_customers.py，可导出客户工作表、订单关联、客户标识观察、地区计数、质量清单与阶段报告。

预期客户记录数和订单数均为 99,441，客户标识数 96,096；地区表 4,310 行，order_count 合计 99,441。控制总额差额全为 0。

当前地区标记均为 0，仍需后续地理表核查邮编覆盖及城市、州一致性。复购指标需另定状态和观察窗口。完成本次整理后进入商品属性与品类翻译表。

## 我的理解

为什么 city_orders 只有 4,310 行，却仍包含 99,441 笔订单？用哪个字段核对？这张地区表的一行代表什么？

我的回答:city_orders['order_count'].sum() == 99441，就说明地区分组后仍能追溯全部订单；每一行代表一个“州＋城市”组合

运行后的观察或疑问：
